In [1]:
import os
import numpy as np
import librosa
import joblib
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import classification_report, confusion_matrix
import tensorflow as tf
from tensorflow.keras.callbacks import EarlyStopping

In [2]:
# =========================
# CONFIG
# =========================
data_dir = r"D:\Basant\Graduation Project\AI\AI-online\Final Data\BALANCED_DATA"

selected_classes = ["Siren", "Gunshot", "Cracking"]

SAMPLE_RATE = 16000
DURATION = 3
CHUNKS = 4

In [3]:
# =========================
# FEATURE EXTRACTION
# =========================
def extract_features(file_path):
    audio, sr = librosa.load(file_path, sr=SAMPLE_RATE)

    target_len = sr * DURATION

    if len(audio) < target_len:
        audio = np.pad(audio, (0, target_len - len(audio)))
    else:
        audio = audio[:target_len]

    max_val = np.max(np.abs(audio))
    if max_val == 0:
        max_val = 1e-9

    audio = audio / max_val

    chunk_size = len(audio) // CHUNKS
    features_list = []

    for i in range(CHUNKS):
        chunk = audio[i * chunk_size:(i + 1) * chunk_size]

        mfcc = librosa.feature.mfcc(y=chunk, sr=sr, n_mfcc=20)

        mfcc_mean = np.mean(mfcc, axis=1)
        mfcc_std = np.std(mfcc, axis=1)

        delta = librosa.feature.delta(mfcc)
        delta_mean = np.mean(delta, axis=1)

        zcr = np.mean(librosa.feature.zero_crossing_rate(chunk))
        centroid = np.mean(librosa.feature.spectral_centroid(y=chunk, sr=sr))
        bandwidth = np.mean(librosa.feature.spectral_bandwidth(y=chunk, sr=sr))
        rolloff = np.mean(librosa.feature.spectral_rolloff(y=chunk, sr=sr))
        rms = np.mean(librosa.feature.rms(y=chunk))

        rolloff_ratio = rolloff / (centroid + 1e-6)

        features = np.concatenate([
            mfcc_mean,
            mfcc_std,
            delta_mean,
            [zcr, centroid, bandwidth, rolloff, rolloff_ratio, rms]
        ])

        features_list.append(features)

    features_list = np.array(features_list)

    return np.concatenate([
        np.mean(features_list, axis=0),
        np.std(features_list, axis=0)
    ])

In [4]:
# =========================
# LOAD DATA
# =========================
X, y = [], []

print("🎧 Loading Dataset...")

for cls in selected_classes:
    class_path = os.path.join(data_dir, cls)

    if not os.path.isdir(class_path):
        continue

    for file in tqdm(os.listdir(class_path), desc=cls):
        path = os.path.join(class_path, file)

        try:
            X.append(extract_features(path))
            y.append(cls)
        except Exception as e:
            print("Skipping:", file)

X = np.array(X)
y = np.array(y)

print("Dataset shape:", X.shape)

🎧 Loading Dataset...


Siren:   0%|          | 0/1000 [00:00<?, ?it/s]c:\Users\LEGEND\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Cracking: 100%|██████████| 1000/1000 [00:42<00:00, 23.63it/s]

Dataset shape: (3000, 132)


In [5]:
# =========================
# LABEL ENCODER
# =========================
le = LabelEncoder()
y = le.fit_transform(y)

print("Classes:", le.classes_)

Classes: ['Cracking' 'Gunshot' 'Siren']


In [6]:
# =========================
# SPLIT
# =========================
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [7]:
# =========================
# SCALING
# =========================
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [8]:
# =========================
# MODEL
# =========================
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(X_train.shape[1],)),

    tf.keras.layers.Dense(256, activation='relu'),
    tf.keras.layers.Dropout(0.3),

    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dropout(0.3),

    tf.keras.layers.Dense(64, activation='relu'),

    tf.keras.layers.Dense(3, activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [9]:
# =========================
# EARLY STOPPING
# =========================
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True
)

In [10]:
# =========================
# TRAIN
# =========================
history = model.fit(
    X_train,
    y_train,
    validation_split=0.2,
    epochs=100,
    batch_size=32,
    callbacks=[early_stop]
)

Epoch 1/100
60/60 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - accuracy: 0.7698 - loss: 0.5214 - val_accuracy: 0.8708 - val_loss: 0.3155
Epoch 2/100
60/60 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.8781 - loss: 0.3149 - val_accuracy: 0.9062 - val_loss: 0.2544
Epoch 3/100
60/60 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.9010 - loss: 0.2627 - val_accuracy: 0.9062 - val_loss: 0.2393
Epoch 4/100
60/60 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9193 - loss: 0.2156 - val_accuracy: 0.9062 - val_loss: 0.2368
Epoch 5/100
60/60 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.9328 - loss: 0.1794 - val_accuracy: 0.9167 - val_loss: 0.2427
Epoch 6/100
60/60 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9365 - loss: 0.1704 - val_accuracy: 0.9167 - val_loss: 0.2308
Epoch 7/100
60/60 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.9443 - loss: 0.1404 - val_accuracy: 0.9125 - val_loss: 0.2677
Epoch 8/100
60/60 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.9526 - loss: 0.1189 - val_accuracy: 0.9208 - v

In [11]:
# =========================
# EVALUATION
# =========================
y_pred_prob = model.predict(X_test)
y_pred = np.argmax(y_pred_prob, axis=1)

print("\n📊 Classification Report:\n")
print(classification_report(y_test, y_pred, target_names=le.classes_))

print("\n📌 Confusion Matrix:\n")
print(confusion_matrix(y_test, y_pred))

19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step

📊 Classification Report:

              precision    recall  f1-score   support

    Cracking       0.93      0.86      0.89       200
     Gunshot       0.88      0.93      0.90       200
       Siren       0.99      0.99      0.99       200

    accuracy                           0.93       600
   macro avg       0.93      0.93      0.93       600
weighted avg       0.93      0.93      0.93       600


📌 Confusion Matrix:

[[173  25   2]
 [ 13 186   1]
 [  1   1 198]]


In [13]:
# =========================
# SAVE
# =========================
os.makedirs("artifacts", exist_ok=True)

model.save("artifacts/type_model.h5")

joblib.dump(scaler, "artifacts/type_scaler.pkl")
joblib.dump(le, "artifacts/type_encoder.pkl")

print("✅ Saved Successfully")

✅ Saved Successfully
